<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 5.2: The ALU, the Immediates and the Registers
**Prev: [The RV32I Instruction Set](5.1_riscv_isa.ipynb)**<br>

## Motivation
A processor's datapath is built from a few blocks that every instruction passes through. This
notebook builds three of them, as modules of the `riscv/` project that the processors of the
next notebooks instantiate:

- the **ALU**, which does the arithmetic and logic of RV32I (your exercise);
- the **immediate generator**, which takes the immediate out of an instruction, whatever its format;
- the **register file**, the 32 registers, from [Module 2.8](2.8_memories.ipynb).

Each is checked against a model written in Scala: the ALU against the golden model of the course,
`riscv.IsaSim` (an RV32I interpreter), and the immediate generator against the software decoder of
[Module 5.1](5.1_riscv_isa.ipynb).

## Setup

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
interp.load.module(os.pwd / "source" / "load-riscv.sc")

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.simulator.PeekPokeAPI._
import riscv._

---
# The ALU

The ALU computes `a op b` for the ten operations of the RV32I arithmetic instructions. They are a
`ChiselEnum` ([Module 2.7](2.7_enums_and_bundles.ipynb)), `riscv.AluOp`; the register-register
instruction of each is its `mnemonic`, and `IsaSim.alu` is the golden model's version of it:

In [ ]:
for (op <- AluOp.all)
  println(f"${AluOp.mnemonic(op)}%-5s 7 op 3 = ${IsaSim.alu(AluOp.mnemonic(op), 7, 3)}")

The I-type instructions (`addi`, `slti`, ...) use the same operations with the immediate as `b`,
which is why there is no `AluOp` for them. Other instructions need an addition too: loads, stores
and `jalr` compute register + immediate, and `auipc` adds the immediate to the pc.

The ALU is parameterized by its `width`: `Alu(32)` is the one of RV32I, but nothing in it is
specific to 32 bits. Its interface:

In [ ]:
println(os.read(os.pwd / "riscv" / "src" / "main" / "scala" / "riscv" / "Alu.scala"))

## Exercise: complete the ALU

Open `riscv/src/main/scala/riscv/Alu.scala` (in Jupyter Lab: the file browser on the left) and
replace the `???` with the logic of every `AluOp`. Some hints:

- a shift uses only the low `log2Ceil(width)` bits of `b` (5 bits for 32), as RV32I says;
- `Slt` compares as signed numbers (`asSInt`), `Sltu` as unsigned ones; both give 1 or 0;
- `Sra` shifts in copies of the sign bit: shift the `SInt`, then turn it back into a `UInt`;
- `MuxLookup` ([Module 2.7](2.7_enums_and_bundles.ipynb)) selects by the operation.

Until then, the next two cells stop with a `NotImplementedError` when they build the ALU: that is
the `???`, not a broken setup. When your ALU is written, **restart the kernel** (the running kernel keeps the classes it loaded) and run the notebook
again from the top: `load-riscv.sc` recompiles the project. The test below compares your ALU with
`IsaSim.alu` on random operands and on the corner cases random numbers rarely hit, in one
simulation: building it (elaborating, then compiling with Verilator) takes most of the time, the
5000 checks a small part of it, so one simulation with many pokes is much faster than many small
ones.

In [ ]:
simulate(new Alu) { c =>
  val rnd              = new scala.util.Random(1)
  def unsigned(x: Int) = BigInt(Integer.toUnsignedLong(x))
  val corners          = Seq(0, 1, -1, 31, 32, Int.MaxValue, Int.MinValue)
  def operand()        = if (rnd.nextInt(4) == 0) corners(rnd.nextInt(corners.size)) else rnd.nextInt()
  for (op <- AluOp.all; _ <- 0 until 500) {
    val (a, b) = (operand(), operand())
    c.io.op.poke(op)
    c.io.a.poke(unsigned(a).U)
    c.io.b.poke(unsigned(b).U)
    c.io.out.expect(unsigned(IsaSim.alu(AluOp.mnemonic(op), a, b)).U, s"${AluOp.mnemonic(op)}($a, $b)")
  }
}
println("SUCCESS!!")

And the parameterization: an 8-bit ALU wraps around at 256, and shifts by the low 3 bits of `b`.

In [ ]:
simulate(new Alu(8)) { c =>
  def check(op: AluOp.Type, a: Int, b: Int, expected: Int) = {
    c.io.op.poke(op)
    c.io.a.poke(a.U)
    c.io.b.poke(b.U)
    c.io.out.expect(expected.U, s"${AluOp.mnemonic(op)}($a, $b)")
  }
  check(AluOp.Add, 200, 100, 44) // 300 - 256
  check(AluOp.Sub, 0, 1, 255)
  check(AluOp.Sll, 1, 9, 2)       // 9 = 0b1001: shift by 1
  check(AluOp.Sra, 0x80, 1, 0xc0) // the sign bit is bit 7
  check(AluOp.Slt, 0xff, 1, 1)    // -1 < 1
  check(AluOp.Sltu, 0xff, 1, 0)
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val shamt = io.b(log2Ceil(width) - 1, 0)
  io.out := MuxLookup(io.op, 0.U)(
    Seq(
      AluOp.Add  -&gt; (io.a + io.b),
      AluOp.Sub  -&gt; (io.a - io.b),
      AluOp.Sll  -&gt; (io.a &lt;&lt; shamt)(width - 1, 0),
      AluOp.Slt  -&gt; (io.a.asSInt &lt; io.b.asSInt),
      AluOp.Sltu -&gt; (io.a &lt; io.b),
      AluOp.Xor  -&gt; (io.a ^ io.b),
      AluOp.Srl  -&gt; (io.a &gt;&gt; shamt),
      AluOp.Sra  -&gt; (io.a.asSInt &gt;&gt; shamt).asUInt,
      AluOp.Or   -&gt; (io.a | io.b),
      AluOp.And  -&gt; (io.a &amp; io.b)
    )
  )
</pre></article></div></section></div>

---
# The immediate generator

[Module 5.1](5.1_riscv_isa.ipynb) showed how each format scatters its immediate over the
instruction. In hardware, that is only wiring: every immediate bit comes from a fixed instruction
bit (or is a copy of the sign, bit 31, or a zero), and a multiplexer picks the format. The decoder
of the next notebook tells it the format:

In [ ]:
println(os.read(os.pwd / "riscv" / "src" / "main" / "scala" / "riscv" / "ImmGen.scala"))

`Fill(n, sign)` is `n` copies of the sign bit and `Cat` concatenates, most significant first, so
each line can be read against the format table of 5.1. The test compares it with the software
decoder, `Rv32i.decode`, on random instructions of every format:

In [ ]:
// random instructions as in 5.1: the pattern's fixed bits, random bits elsewhere (the immediate)
val formats = Seq(
  ImmFormat.I -> Seq("ADDI", "LW", "JALR"),
  ImmFormat.S -> Seq("SW"),
  ImmFormat.B -> Seq("BEQ"),
  ImmFormat.U -> Seq("LUI", "AUIPC"),
  ImmFormat.J -> Seq("JAL")
)
simulate(new ImmGen) { c =>
  val rnd = new scala.util.Random(1)
  for ((format, names) <- formats; name <- names; _ <- 0 until 100) {
    val word = Instructions.random(name, rnd)
    c.io.inst.poke(word.U)
    c.io.format.poke(format)
    c.io.imm.expect(BigInt(Integer.toUnsignedLong(Rv32i.decode(word).get.imm)).U, f"$name 0x$word%08x")
  }
}
println("SUCCESS!!")

---
# The register file

`riscv.RegFile` is the same register file as the one of [Module 2.8](2.8_memories.ipynb): two
combinational read ports (an instruction reads `rs1` and `rs2` in the same cycle), one write port
that writes at the clock edge, `x0` hard-wired to 0, and a debug port that lets a test read any
register without disturbing the processor. The processors read their results through it.

In [ ]:
println(os.read(os.pwd / "riscv" / "src" / "main" / "scala" / "riscv" / "RegFile.scala"))

In [ ]:
simulate(new RegFile) { c =>
  // write x1..x31, and x0 (which must ignore it)
  c.io.wen.poke(true.B)
  for (r <- 0 until 32) {
    c.io.rd.poke(r.U)
    c.io.wdata.poke((100 + r).U)
    c.clock.step()
  }
  c.io.wen.poke(false.B)
  for (r <- 0 until 32) {
    c.io.dbgAddr.poke(r.U)
    c.io.dbgData.expect((if (r == 0) 0 else 100 + r).U, s"x$r")
  }
  // a read in the same cycle as a write to the same register sees the old value
  c.io.rs1.poke(5.U)
  c.io.rd.poke(5.U)
  c.io.wdata.poke(7.U)
  c.io.wen.poke(true.B)
  c.io.rs1Data.expect(105.U)
  c.clock.step()
  c.io.rs1Data.expect(7.U)
}
println("SUCCESS!!")

That last check matters for pipelines: an instruction that reads a register in the same cycle as
an older instruction writes it gets the old value: the new one is written a cycle too late. A
pipelined processor has to deal with it, by forwarding the new value or by waiting.

---
# You're done!

The next notebook builds the decoder, which turns an instruction into the control signals of these
blocks (`AluOp`, `ImmFormat`, register write, ...).

[Return to the top.](#top)